# PathBSR Evaluation Protocol Tables

This notebook renders the **final PathBSR test artifact** into paper-ready evaluation tables. It does not rebuild the model, select hyperparameters, or rerun the test split.

Two protocols are reported:

1. **Primary protocol**: filtered, bidirectional, average-tie, full-entity ranking.
2. **Auxiliary comparison**: filtered, tail-only, optimistic-tie, full-entity ranking.

The auxiliary table is not the PathBSR primary result and must not be presented as protocol-comparable unless the compared work uses the same direction and tie policy. Model definitions follow the current defaults in `src/pathbsr/config.py`.


In [ ]:
from __future__ import annotations

import csv
import math
import statistics
from pathlib import Path

try:
    from IPython.display import Markdown, display
except ImportError:
    Markdown = None
    display = None

PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / 'pyproject.toml').is_file():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / 'pyproject.toml').is_file():
    raise FileNotFoundError('Could not locate the PathBSR repository root')

FINAL_CSV = PROJECT_ROOT / 'results/runs/pathbsr_best_model_test.csv'
OUTPUT_PATH = PROJECT_ROOT / 'results/evaluation_protocol/pathbsr_protocol_score_tables.csv'

DATASET_ORDER = [
    'FB15K-237-10',
    'FB15K-237-20',
    'FB15K-237-50',
    'NELL23K',
    'WD-singer',
    'WN18RR',
]

DEFAULT_CONFIG_EXPECTATIONS = {
    'rule_library_topk': 128,
}

def show_markdown(text: str) -> None:
    if Markdown is not None and display is not None:
        display(Markdown(text))
    else:
        print(text)

print('PROJECT_ROOT:', PROJECT_ROOT)
print('FINAL_CSV:', FINAL_CSV.relative_to(PROJECT_ROOT))


## Artifact and protocol validation

The final CSV is checked directly. The checks below prevent silently mixing official and de-overlapped WD-singer results or mislabelling tail-only optimistic scores as the primary protocol.


In [ ]:
if not FINAL_CSV.is_file():
    raise FileNotFoundError(
        f'{FINAL_CSV} not found. Regenerate it with scripts/run_pathbsr.py '
        '--split test --output results/runs/pathbsr_best_model_test.csv'
    )

with FINAL_CSV.open('r', encoding='utf-8', newline='') as handle:
    results = list(csv.DictReader(handle))

required_columns = {
    'dataset',
    'split',
    'evaluation_variant',
    'mrr',
    'hits@1',
    'hits@3',
    'hits@10',
    'tailopt_mrr',
    'tailopt_hits@1',
    'tailopt_hits@3',
    'tailopt_hits@10',
    'tail_mrr',
    'head_mrr',
    'rule_library_topk',
}
missing_columns = sorted(required_columns.difference(results[0].keys() if results else []))
if missing_columns:
    raise ValueError(f'Final CSV is missing required columns: {missing_columns}')

by_dataset = {row['dataset']: row for row in results}
missing = [dataset for dataset in DATASET_ORDER if dataset not in by_dataset]
extra = sorted(set(by_dataset).difference(DATASET_ORDER))
if missing:
    raise ValueError(f'Missing datasets in final CSV: {missing}')
if extra:
    raise ValueError(f'Unexpected datasets in final CSV: {extra}')

for row in results:
    if row['split'] != 'test':
        raise ValueError(f"{row['dataset']} has split={row['split']!r}; expected 'test'")
    if row['evaluation_variant'] != 'official':
        raise ValueError(f"{row['dataset']} uses evaluation_variant={row['evaluation_variant']!r}; expected 'official'")
    if int(float(row['rule_library_topk'])) != DEFAULT_CONFIG_EXPECTATIONS['rule_library_topk']:
        raise ValueError(f"{row['dataset']} has unexpected rule_library_topk={row['rule_library_topk']}")

results = [by_dataset[dataset] for dataset in DATASET_ORDER]
show_markdown('Protocol CSV checks passed. The table uses official test rows only.')


## Protocol definitions

For the primary protocol, the filtered average-tie rank is

\[
\operatorname{rank}_{\mathrm{avg}} = N_{>} + \frac{N_{=}+1}{2},
\]

and both tail and reverse-relation head queries are included. The auxiliary tail-only table uses optimistic rank, (N_{>}+1), and is reported only as a separately labelled comparison.


In [ ]:
def markdown_table(headers: list[str], rows: list[list[str]]) -> str:
    lines = [
        '| ' + ' | '.join(headers) + ' |',
        '| ' + ' | '.join(['---'] * len(headers)) + ' |',
    ]
    lines.extend('| ' + ' | '.join(str(value) for value in row) + ' |' for row in rows)
    return '\n'.join(lines)

show_markdown(markdown_table(
    ['Role', 'Directions', 'Filtered', 'Tie policy', 'Candidate set'],
    [
        ['Primary', 'Tail + head', 'Yes', 'Average tie', 'All entities'],
        ['Auxiliary comparison', 'Tail only', 'Yes', 'Optimistic', 'All entities'],
    ],
))


## Primary test results

Filtered, bidirectional, average-tie, full-entity ranking on the official test splits. The macro row is an unweighted average over the six datasets.


In [ ]:
PRIMARY_METRICS = [('mrr', 'MRR'), ('hits@1', 'Hits@1'), ('hits@3', 'Hits@3'), ('hits@10', 'Hits@10')]
primary_rows = [
    [
        row['dataset'],
        *[f"{float(row[source]):.6f}" for source, _ in PRIMARY_METRICS],
        f"{int(row['num_queries']):,}",
    ]
    for row in results
]
primary_rows.append([
    'Macro average',
    *[f"{statistics.fmean(float(row[source]) for row in results):.6f}" for source, _ in PRIMARY_METRICS],
    f"{sum(int(row['num_queries']) for row in results):,}",
])
show_markdown(markdown_table(
    ['Dataset', *[label for _, label in PRIMARY_METRICS], 'Queries'],
    primary_rows,
))


## Auxiliary tail-only optimistic comparison

These values reuse the same score vectors but change both the evaluated direction and tie policy. They must remain separate from the primary PathBSR table.


In [ ]:
AUXILIARY_METRICS = [
    ('tailopt_mrr', 'MRR'),
    ('tailopt_hits@1', 'Hits@1'),
    ('tailopt_hits@3', 'Hits@3'),
    ('tailopt_hits@10', 'Hits@10'),
]
auxiliary_rows = [
    [
        row['dataset'],
        *[f"{float(row[source]):.6f}" for source, _ in AUXILIARY_METRICS],
        f"{int(row['tailopt_num_queries']):,}",
    ]
    for row in results
]
auxiliary_rows.append([
    'Macro average',
    *[f"{statistics.fmean(float(row[source]) for row in results):.6f}" for source, _ in AUXILIARY_METRICS],
    f"{sum(int(row['tailopt_num_queries']) for row in results):,}",
])
show_markdown(markdown_table(
    ['Dataset', *[label for _, label in AUXILIARY_METRICS], 'Queries'],
    auxiliary_rows,
))


## Tail/head breakdown of the primary protocol


In [ ]:
BREAKDOWN_METRICS = [
    ('tail_mrr', 'Tail MRR'),
    ('tail_hits@1', 'Tail Hits@1'),
    ('tail_hits@3', 'Tail Hits@3'),
    ('tail_hits@10', 'Tail Hits@10'),
    ('head_mrr', 'Head MRR'),
    ('head_hits@1', 'Head Hits@1'),
    ('head_hits@3', 'Head Hits@3'),
    ('head_hits@10', 'Head Hits@10'),
]
breakdown_rows = [
    [row['dataset'], *[f"{float(row[source]):.6f}" for source, _ in BREAKDOWN_METRICS]]
    for row in results
]
show_markdown(markdown_table(
    ['Dataset', *[label for _, label in BREAKDOWN_METRICS]],
    breakdown_rows,
))


## Export derived protocol tables

The exported CSV is a derived presentation artifact. The final CSV remains the authoritative result source.


In [ ]:
export_table = []
for row in results:
    export_table.append({
        'dataset': row['dataset'],
        'split': row['split'],
        'protocol_id': 'primary_bidirectional_average_tie',
        'mrr': row['mrr'],
        'hits@1': row['hits@1'],
        'hits@3': row['hits@3'],
        'hits@10': row['hits@10'],
        'num_queries': row['num_queries'],
    })
for row in results:
    export_table.append({
        'dataset': row['dataset'],
        'split': row['split'],
        'protocol_id': 'auxiliary_tail_only_optimistic',
        'mrr': row['tailopt_mrr'],
        'hits@1': row['tailopt_hits@1'],
        'hits@3': row['tailopt_hits@3'],
        'hits@10': row['tailopt_hits@10'],
        'num_queries': row['tailopt_num_queries'],
    })

OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
fieldnames = ['dataset', 'split', 'protocol_id', 'mrr', 'hits@1', 'hits@3', 'hits@10', 'num_queries']
with OUTPUT_PATH.open('w', encoding='utf-8', newline='') as handle:
    writer = csv.DictWriter(handle, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(export_table)
print('Saved:', OUTPUT_PATH.relative_to(PROJECT_ROOT))
show_markdown(markdown_table(
    fieldnames,
    [[row[name] for name in fieldnames] for row in export_table],
))
